# Assignment Sesi 29 Tugas 2
Nama: Faraday Barr Fatahillah

## Pipeline Overview
1. Parse individual brochure PDFs
2. Combine all into one PDF (with title separators)
3. Parse combined PDF
4. Recursive chunking
5. Local embeddings via SentenceTransformer
6. Store in ChromaDB (local) & Pinecone (cloud)
7. Hybrid Search = Dense (embedding) + Sparse (BM25)
8. Run 3 queries

## 0. Install Dependencies

In [ ]:
!pip install -q \
    pymupdf \
    pdfplumber \
    reportlab \
    sentence-transformers \
    chromadb \
    pinecone-client \
    rank_bm25 \
    numpy \
    tqdm

## 1. Configuration

In [ ]:
import os

# ── Folder containing the Indonesian Mitsubishi PDF brochures ──────────────────
PDF_FOLDER = "/Session29_Tugas2_PDFS"   # <-- adjust if needed

# ── Output paths ──────────────────────────────────────────────────────────────
COMBINED_PDF_PATH = "/tmp/mitsubishi_combined.pdf"
CHROMA_PERSIST_DIR = "/tmp/chroma_mitsubishi"
CHROMA_COLLECTION = "mitsubishi_id"

# ── Pinecone ───────────────────────────────────────────────────────────────────
PINECONE_API_KEY  = os.getenv("PINECONE_API_KEY", "YOUR_PINECONE_API_KEY")
PINECONE_ENV      = os.getenv("PINECONE_ENV",     "us-east-1")          # e.g. 'us-east-1'
PINECONE_INDEX    = "mitsubishi-id"

# ── Embedding model (runs locally, no API key needed) ─────────────────────────
EMBED_MODEL_NAME  = "paraphrase-multilingual-MiniLM-L12-v2"   # supports Bahasa Indonesia
EMBED_DIM         = 384

# ── Chunking ───────────────────────────────────────────────────────────────────
CHUNK_SIZE        = 500   # characters
CHUNK_OVERLAP     = 100

# ── Hybrid search weights ──────────────────────────────────────────────────────
ALPHA             = 0.6   # 0 = pure BM25, 1 = pure dense
TOP_K             = 5

print("Configuration ready.")

## 2. Parse Individual Brochure PDFs

In [ ]:
import pymupdf          # PyMuPDF
import pdfplumber
import re
from pathlib import Path


def extract_text_from_pdf(pdf_path: str) -> str:
    """
    Extract text + tables from a single PDF.
    Uses PyMuPDF for body text, pdfplumber for tables.
    """
    # ── Body text ──────────────────────────────────────────────────────────────
    doc = pymupdf.open(pdf_path)
    pages_text = []
    for page in doc:
        text = page.get_text("text").strip()
        if text:
            pages_text.append(text)
    doc.close()

    # ── Tables ─────────────────────────────────────────────────────────────────
    tables_text = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            for table in page.extract_tables():
                for row in table:
                    clean = [str(cell).strip() if cell else "" for cell in row]
                    if any(clean):
                        tables_text.append(" | ".join(clean))

    combined = "\n".join(pages_text)
    if tables_text:
        combined += "\n\n--- TABLES ---\n" + "\n".join(tables_text)

    return re.sub(r"\n{3,}", "\n\n", combined).strip()


# ── Discover PDFs ──────────────────────────────────────────────────────────────
pdf_files = sorted(Path(PDF_FOLDER).glob("*.pdf"))
print(f"Found {len(pdf_files)} PDF(s):")
for p in pdf_files:
    print(f"  • {p.name}")

In [ ]:
from tqdm import tqdm

# Extract text from every brochure
brochures: list[dict] = []   # [{"title": str, "path": str, "text": str}]

for pdf_path in tqdm(pdf_files, desc="Extracting PDFs"):
    title = pdf_path.stem          # filename without extension = brochure title
    text  = extract_text_from_pdf(str(pdf_path))
    brochures.append({"title": title, "path": str(pdf_path), "text": text})
    print(f"  [{title}] → {len(text):,} chars")

print(f"\nTotal brochures parsed: {len(brochures)}")

## 3. Combine All Brochures into One PDF (with title separators)

In [ ]:
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, PageBreak, HRFlowable
)
from reportlab.lib import colors


def build_combined_pdf(brochures: list[dict], output_path: str) -> None:
    """
    Creates one PDF that contains all brochure texts.
    Each brochure starts with a bold title separator page.
    """
    doc    = SimpleDocTemplate(
        output_path,
        pagesize=A4,
        rightMargin=2*cm, leftMargin=2*cm,
        topMargin=2*cm,   bottomMargin=2*cm,
    )
    styles = getSampleStyleSheet()

    title_style = ParagraphStyle(
        "BrochureTitle",
        parent=styles["Heading1"],
        fontSize=18,
        textColor=colors.HexColor("#C8102E"),   # Mitsubishi red
        spaceAfter=12,
    )
    body_style = ParagraphStyle(
        "BrochureBody",
        parent=styles["Normal"],
        fontSize=9,
        leading=13,
        spaceAfter=4,
    )

    story = []
    for i, brochure in enumerate(brochures):
        if i > 0:
            story.append(PageBreak())

        # ── Title separator ────────────────────────────────────────────────────
        story.append(HRFlowable(width="100%", thickness=2, color=colors.HexColor("#C8102E")))
        story.append(Spacer(1, 6))
        story.append(Paragraph(f"BROCHURE: {brochure['title']}", title_style))
        story.append(HRFlowable(width="100%", thickness=1, color=colors.grey))
        story.append(Spacer(1, 12))

        # ── Body text (escape XML special chars for ReportLab) ─────────────────
        safe_text = (
            brochure["text"]
            .replace("&", "&amp;")
            .replace("<", "&lt;")
            .replace(">", "&gt;")
        )
        for para in safe_text.split("\n\n"):
            para = para.strip()
            if para:
                story.append(Paragraph(para.replace("\n", "<br/>"), body_style))
                story.append(Spacer(1, 4))

    doc.build(story)
    print(f"Combined PDF saved → {output_path}")


build_combined_pdf(brochures, COMBINED_PDF_PATH)

## 4. Parse Combined PDF

In [ ]:
combined_text = extract_text_from_pdf(COMBINED_PDF_PATH)
print(f"Combined PDF total characters: {len(combined_text):,}")
print("\n--- Preview (first 500 chars) ---")
print(combined_text[:500])

## 5. Recursive Chunking

In [ ]:
def recursive_split(
    text: str,
    chunk_size: int = CHUNK_SIZE,
    chunk_overlap: int = CHUNK_OVERLAP,
    separators: list[str] | None = None,
) -> list[str]:
    """
    Recursively split text using a priority list of separators.
    Falls back to character-level slicing when no separator works.
    """
    if separators is None:
        separators = ["\n\n", "\n", ". ", " ", ""]

    def _split(text: str, seps: list[str]) -> list[str]:
        if len(text) <= chunk_size:
            return [text] if text.strip() else []

        sep = seps[0] if seps else ""
        parts = text.split(sep) if sep else list(text)

        chunks: list[str] = []
        current = ""
        for part in parts:
            candidate = (current + sep + part) if current else part
            if len(candidate) <= chunk_size:
                current = candidate
            else:
                if current.strip():
                    if len(current) > chunk_size and len(seps) > 1:
                        chunks.extend(_split(current, seps[1:]))
                    else:
                        chunks.append(current)
                current = part
        if current.strip():
            if len(current) > chunk_size and len(seps) > 1:
                chunks.extend(_split(current, seps[1:]))
            else:
                chunks.append(current)
        return chunks

    raw_chunks = _split(text, separators)

    # Apply overlap
    if chunk_overlap == 0 or len(raw_chunks) <= 1:
        return raw_chunks

    overlapped: list[str] = [raw_chunks[0]]
    for i in range(1, len(raw_chunks)):
        tail  = overlapped[-1][-chunk_overlap:]   # last N chars of previous chunk
        merged = (tail + " " + raw_chunks[i]).strip()
        overlapped.append(merged[:chunk_size + chunk_overlap])
    return overlapped


# ── Chunk combined text ────────────────────────────────────────────────────────
all_chunks = recursive_split(combined_text)
print(f"Total chunks: {len(all_chunks)}")
print(f"Avg chunk length: {sum(len(c) for c in all_chunks)/len(all_chunks):.0f} chars")
print("\n--- Sample chunk ---")
print(all_chunks[0])

In [ ]:
# Attach source brochure metadata to each chunk
# We detect which brochure a chunk belongs to by looking for the title separator pattern

def tag_chunks_with_source(
    chunks: list[str],
    brochures: list[dict],
) -> list[dict]:
    """
    Returns [{"id": str, "text": str, "source": str}, ...]
    The source is inferred by scanning backwards for the last brochure title.
    """
    # Build position map from combined_text
    current_source = "unknown"
    tagged: list[dict] = []

    for i, chunk in enumerate(chunks):
        # Check if this chunk contains a brochure title marker
        for brochure in brochures:
            if f"BROCHURE: {brochure['title']}" in chunk:
                current_source = brochure["title"]
                break
        tagged.append({
            "id":     f"chunk_{i:05d}",
            "text":   chunk.strip(),
            "source": current_source,
        })
    return tagged


tagged_chunks = tag_chunks_with_source(all_chunks, brochures)
print(f"Tagged {len(tagged_chunks)} chunks.")

# Quick distribution
from collections import Counter
dist = Counter(c["source"] for c in tagged_chunks)
for src, count in dist.most_common():
    print(f"  {src}: {count} chunks")

## 6. Local Embeddings with SentenceTransformer

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

print(f"Loading embedding model: {EMBED_MODEL_NAME} ...")
embed_model = SentenceTransformer(EMBED_MODEL_NAME)

texts = [c["text"] for c in tagged_chunks]

print(f"Embedding {len(texts)} chunks (this may take a minute) ...")
embeddings = embed_model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,   # L2-normalize for cosine similarity
)

print(f"Embeddings shape: {embeddings.shape}")

## 7a. Store in ChromaDB (local vector DB)

In [ ]:
import chromadb
from chromadb.config import Settings

chroma_client = chromadb.PersistentClient(
    path=CHROMA_PERSIST_DIR,
    settings=Settings(anonymized_telemetry=False),
)

# Drop & recreate for a clean slate
try:
    chroma_client.delete_collection(CHROMA_COLLECTION)
except Exception:
    pass

collection = chroma_client.create_collection(
    name=CHROMA_COLLECTION,
    metadata={"hnsw:space": "cosine"},
)

# Upsert in batches of 100
BATCH = 100
for start in tqdm(range(0, len(tagged_chunks), BATCH), desc="ChromaDB upsert"):
    batch = tagged_chunks[start : start + BATCH]
    collection.add(
        ids        = [c["id"]   for c in batch],
        documents  = [c["text"] for c in batch],
        embeddings = embeddings[start : start + BATCH].tolist(),
        metadatas  = [{"source": c["source"]} for c in batch],
    )

print(f"ChromaDB collection '{CHROMA_COLLECTION}' → {collection.count()} documents")

## 7b. Store in Pinecone (cloud vector DB)

In [ ]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=PINECONE_API_KEY)

# Create index if it doesn't exist
existing_indexes = [idx.name for idx in pc.list_indexes()]
if PINECONE_INDEX not in existing_indexes:
    pc.create_index(
        name      = PINECONE_INDEX,
        dimension = EMBED_DIM,
        metric    = "cosine",
        spec      = ServerlessSpec(cloud="aws", region=PINECONE_ENV),
    )
    print(f"Created Pinecone index '{PINECONE_INDEX}'")
else:
    print(f"Using existing Pinecone index '{PINECONE_INDEX}'")

index = pc.Index(PINECONE_INDEX)

# Upsert vectors in batches
BATCH = 100
for start in tqdm(range(0, len(tagged_chunks), BATCH), desc="Pinecone upsert"):
    batch  = tagged_chunks[start : start + BATCH]
    embeds = embeddings[start : start + BATCH]
    vectors = [
        {
            "id":       chunk["id"],
            "values":   emb.tolist(),
            "metadata": {"text": chunk["text"], "source": chunk["source"]},
        }
        for chunk, emb in zip(batch, embeds)
    ]
    index.upsert(vectors=vectors)

stats = index.describe_index_stats()
print(f"Pinecone index stats: {stats}")

## 8. BM25 Index (Sparse Retrieval)

In [ ]:
from rank_bm25 import BM25Okapi

def tokenize(text: str) -> list[str]:
    """Simple whitespace + lowercase tokenizer (works for Bahasa Indonesia)."""
    return re.sub(r"[^\w\s]", " ", text.lower()).split()


corpus_tokens = [tokenize(c["text"]) for c in tagged_chunks]
bm25 = BM25Okapi(corpus_tokens)
print(f"BM25 index built over {len(corpus_tokens)} documents.")

## 9. Hybrid Search Function (Dense + BM25)

In [ ]:
def min_max_norm(arr: np.ndarray) -> np.ndarray:
    """Normalize array to [0, 1]."""
    lo, hi = arr.min(), arr.max()
    return (arr - lo) / (hi - lo + 1e-9)


def hybrid_search(
    query: str,
    top_k: int = TOP_K,
    alpha: float = ALPHA,
    use_pinecone: bool = False,
) -> list[dict]:
    """
    Hybrid search: score = alpha * dense_score + (1-alpha) * bm25_score

    Args:
        query        : user query string
        top_k        : number of results to return
        alpha        : weight for dense (embedding) scores [0..1]
        use_pinecone : if True, fetch dense scores from Pinecone;
                       otherwise use ChromaDB (default)
    Returns:
        list of dicts with keys: rank, id, source, score, text
    """
    n = len(tagged_chunks)

    # ── 1. Dense scores ────────────────────────────────────────────────────────
    query_emb = embed_model.encode(
        [query], normalize_embeddings=True
    )[0]

    if use_pinecone:
        # Pinecone returns only top-k; we need scores for ALL chunks → use ChromaDB
        # (Pinecone is used for upsert; full-corpus scoring is via ChromaDB/numpy)
        pass  # fall through to numpy path

    # Dot product against all embeddings (cosine; already normalized)
    dense_scores_raw = embeddings @ query_emb           # shape (n,)
    dense_scores = min_max_norm(dense_scores_raw)

    # ── 2. BM25 scores ─────────────────────────────────────────────────────────
    query_tokens    = tokenize(query)
    bm25_scores_raw = np.array(bm25.get_scores(query_tokens))
    bm25_scores     = min_max_norm(bm25_scores_raw)

    # ── 3. Fusion ──────────────────────────────────────────────────────────────
    hybrid_scores = alpha * dense_scores + (1 - alpha) * bm25_scores

    # ── 4. Top-k results ───────────────────────────────────────────────────────
    top_indices = np.argsort(hybrid_scores)[::-1][:top_k]

    results = []
    for rank, idx in enumerate(top_indices, start=1):
        chunk = tagged_chunks[idx]
        results.append({
            "rank":         rank,
            "id":           chunk["id"],
            "source":       chunk["source"],
            "hybrid_score": float(hybrid_scores[idx]),
            "dense_score":  float(dense_scores_raw[idx]),
            "bm25_score":   float(bm25_scores_raw[idx]),
            "text":         chunk["text"],
        })
    return results


def pretty_print_results(query: str, results: list[dict]) -> None:
    sep = "=" * 80
    print(f"\n{sep}")
    print(f"QUERY: {query}")
    print(sep)
    for r in results:
        print(f"\n  [Rank {r['rank']}] Source: {r['source']}")
        print(f"  Hybrid={r['hybrid_score']:.4f}  Dense={r['dense_score']:.4f}  BM25={r['bm25_score']:.4f}")
        print(f"  ID: {r['id']}")
        print(f"  Text preview: {r['text'][:300].replace(chr(10), ' ')} ...")
    print()


print("Hybrid search function ready.")

## 10. Run Queries

In [ ]:
# ── Query 1 ────────────────────────────────────────────────────────────────────
Q1 = "Detail spesifikasi Mitsubishi Destinator"
results_q1 = hybrid_search(Q1, top_k=TOP_K)
pretty_print_results(Q1, results_q1)

In [ ]:
# ── Query 2 ────────────────────────────────────────────────────────────────────
Q2 = "Mobil yang cocok untuk Travel dengan jumlah bangku atau seating capacity besar"
results_q2 = hybrid_search(Q2, top_k=TOP_K)
pretty_print_results(Q2, results_q2)

In [ ]:
# ── Query 3 ────────────────────────────────────────────────────────────────────
Q3 = "Mobil untuk perjalanan jauh yang nyaman"
results_q3 = hybrid_search(Q3, top_k=TOP_K)
pretty_print_results(Q3, results_q3)

## 11. Summary of Retrieved Results

In [ ]:
import json

summary = {
    "Q1 - Spesifikasi Destinator":  [{"rank": r["rank"], "source": r["source"], "score": round(r["hybrid_score"], 4)} for r in results_q1],
    "Q2 - Seating Capacity Besar": [{"rank": r["rank"], "source": r["source"], "score": round(r["hybrid_score"], 4)} for r in results_q2],
    "Q3 - Perjalanan Jauh Nyaman": [{"rank": r["rank"], "source": r["source"], "score": round(r["hybrid_score"], 4)} for r in results_q3],
}

print(json.dumps(summary, indent=2, ensure_ascii=False))